# Baseline Model II: PCA Residual Statistical Arbitrage with Kalman-Filtered Fair Value

---

### 1. Motivation and relation to Baseline I

Baseline I followed the standard PCA-statistical-arbitrage structure: rolling factor extraction, regression-based residual isolation, construction of cumulative residual spreads, and static Ornstein–Uhlenbeck normalization. In that specification, the equilibrium level was effectively static within each rolling window, the signal was based on a cross-sectionally adjusted OU equilibrium, and the strategy entered when residual dislocations were sufficiently far from that estimated equilibrium. Baseline II keeps the same economic backbone, but changes the measurement of fair value. Instead of treating equilibrium as a fixed quantity implied solely by the rolling OU fit, it treats fair value as a latent, time-varying state updated sequentially through a Kalman filter. In that sense, Baseline II is not a different trading philosophy; it is a refinement of Baseline I’s estimation problem.

The key methodological shift is therefore the following. Baseline I asks: given a rolling window, where is the spread’s equilibrium? Baseline II asks: given the same rolling structure, how should that equilibrium be updated today, after observing the newest residual information? This distinction matters because in statistical arbitrage the edge does not come only from identifying mean reversion, but from estimating the current center of reversion more accurately than a static model can.

---

### 2. Data, return construction, and rolling estimation

The strategy is implemented on a large universe of liquid technology, semiconductor, software, and infrastructure equities. Prices are downloaded as auto-adjusted closes, and daily simple returns are constructed as

$$
r_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1.
$$

At each date $t$, the model forms a rolling estimation window of length $L_{PCA} = 252$ trading days. Only securities with complete observations in that window are retained, so that the subsequent factor decomposition is performed on a balanced panel at each re-estimation step. In the final implementation, the PCA dimension is $K = 9$, and the shorter regression / spread-estimation window is $L_{OU} = 160$.

---

### 3. Factor extraction by rolling PCA

Let

$$
R_t \in \mathbb{R}^{L_{PCA} \times N_t}
$$

denote the matrix of rolling returns for the valid cross-section at date $t$. The code first standardizes each column:

$$
Y_{i,\tau} = \frac{r_{i,\tau} - \mu_i}{\sigma_i},
$$

where $\mu_i$ and $\sigma_i$ are the rolling mean and standard deviation of stock $i$ over the 252-day window. From these standardized returns, it computes the correlation matrix

$$
\Sigma_t = \mathrm{corr}(Y_t),
$$

then performs an eigenvalue decomposition

$$
\Sigma_t = V_t \Lambda_t V_t^\top.
$$

The first $K = 9$ eigenvectors are retained. If $V_t^{(K)}$ denotes the $N_t \times K$ matrix of leading eigenvectors and

$$
D_\sigma = \mathrm{diag}(\sigma_1, \dots, \sigma_{N_t}),
$$

then the code constructs factor loadings as

$$
Q_t = D_\sigma^{-1} V_t^{(K)},
$$

implemented row-wise as `Q_load = V / sigma[:, None]`, and factor realizations as

$$
F_t = R_t Q_t.
$$

Thus, PCA removes the dominant common variation in the universe and provides a low-dimensional factor space against which each stock can be residualized.

---

### 4. Residual isolation and spread construction

For each stock $i$, the model keeps the most recent $L_{OU} = 160$ observations of both stock returns and PCA factor returns, and estimates the rolling factor model

$$
r_{i,\tau} = \alpha_i + \beta_i^\top F_\tau + \varepsilon_{i,\tau}, \quad \tau = t - L_{OU} + 1, \dots, t.
$$

The quantity of interest is not the raw residual return $\varepsilon_{i,t}$, but the cumulative residual spread

$$
X_{i,t} = X_{i,t-1} + \varepsilon_{i,t}.
$$

This is exactly the same economic move as in Baseline I: rather than trade one-period residual noise, the model trades a residual level process whose dynamics are more compatible with mean-reversion analysis. The difference is that Baseline II no longer treats the center of this spread as static inside the window.

---

### 5. Mean-only Kalman filter for dynamic fair value

The central innovation in Baseline II is the replacement of the static equilibrium estimate by a local-level Kalman filter applied to the spread level $X_{i,t}$. The state variable is the latent fair value $m_{i,t}$. Denoting the previous posterior variance by $P_{i,t-1}$, the code forms the prior variance

$$
R_{i,t} = \frac{P_{i,t-1}}{\delta_A},
$$

where $\delta_A = 0.9982$ is the discount factor controlling how slowly or quickly the filtered mean adapts. Observation noise is estimated from the current rolling regression residuals. If $\widehat{\mathrm{Var}}(\varepsilon_i)$ is the sample residual variance over the regression window, then the implementation sets

$$
\mathrm{ObsVar}_{i,t} = \widehat{\mathrm{Var}}(\varepsilon_i) \cdot L_{OU} \cdot 0.5.
$$

The innovation variance is therefore

$$
Q_{i,t} = R_{i,t} + \mathrm{ObsVar}_{i,t},
$$

with scalar Kalman gain

$$
K_{i,t} = \frac{R_{i,t}}{Q_{i,t}}.
$$

The filtered fair value is then updated by

$$
m_{i,t} = m_{i,t-1} + K_{i,t}(X_{i,t} - m_{i,t-1}),
$$

and the posterior variance evolves as

$$
P_{i,t} = (1 - K_{i,t}) R_{i,t}.
$$

Conceptually, this means that fair value is allowed to move, but only as much as warranted by the new information content of the observed spread innovation. This is the major distinction from Baseline I, where the equilibrium level was effectively pinned down by a static OU estimate inside each rolling window.

---

### 6. Static OU characterization of mean reversion

Although fair value is dynamic, the model still characterizes spread mean reversion through a rolling AR(1) approximation on the cumulative residual history:

$$
X_{i,\tau+1} = a_i + b_i X_{i,\tau} + \eta_{i,\tau+1}.
$$

From this, the code computes the innovation volatility $\sigma_{\eta,i}$, the OU-equivalent equilibrium volatility

$$
\sigma_{eq,i} = \frac{\sigma_{\eta,i}}{\sqrt{1 - b_i^2}},
$$

and the annualized mean-reversion speed

$$
\kappa_i = -\log(b_i) \cdot 252.
$$

Hence, Baseline II still retains the OU interpretation of spread dynamics, but the role of the OU block is different from Baseline I. In Baseline I, the OU fit directly produced the equilibrium-based signal. In Baseline II, OU primarily supplies a structural characterization of spread dynamics, while the signal numerator is generated by the Kalman-updated fair value.

---

### 7. Signal construction

Once the spread $X_{i,t}$ and filtered fair value $m_{i,t}$ are available, the instantaneous deviation is

$$
d_{i,t} = X_{i,t} - m_{i,t}.
$$

This deviation is then standardized. The final implementation does not use only the static OU equilibrium volatility in the denominator. Instead, it uses a dynamic volatility proxy based on the recent realized variation of the spread:

$$
\sigma_{dyn,i,t} =
\begin{cases}
\mathrm{Std}(X_{i,t-59}, \dots, X_{i,t}), & \text{if at least 60 observations are available} \\
\sigma_{eq,i}, & \text{otherwise}
\end{cases}
$$

The trading score is therefore

$$
s_{i,t} = \frac{d_{i,t}}{\sigma_{dyn,i,t}}.
$$

This is another major upgrade over Baseline I. In Baseline I, the signal was essentially a static equilibrium-distance statistic. In Baseline II, the signal is a distance-from-dynamic-fair-value statistic, normalized by an adaptive volatility estimate. That gives the model two dimensions of flexibility: a moving center and a moving scale.

---

### 8. Trading rule and portfolio construction

The position logic is threshold-based. If the stock is flat, the model enters long when

$$
s_{i,t} < -\theta_{in},
$$

and enters short when

$$
s_{i,t} > \theta_{in},
$$

with final parameter $\theta_{in} = 1.5$. Long positions are closed when

$$
s_{i,t} > -\theta_{L,out},
$$

with $\theta_{L,out} = 0.7$, while short positions are closed when

$$
s_{i,t} < \theta_{S,out},
$$

with $\theta_{S,out} = 0.1$. The asymmetric exits reflect the empirical observation that long and short residual dislocations may decay at different speeds.

Across all active names, the stock-level book is equally weighted. If there are $n_t$ active positions, each active stock receives gross stock weight magnitude

$$
\frac{1}{n_t}.
$$

The portfolio then computes its aggregate factor exposure,

$$
\Gamma_t = \sum_{i \in A_t} w_{i,t} \beta_i,
$$

and introduces synthetic hedge legs

$$
w^{(F)}_{k,t} = -\Gamma_{k,t}, \quad k = 1, \dots, K.
$$

Finally, the entire combined stock-plus-factor portfolio is rescaled to a fixed gross exposure,

$$
\sum_j |w_{j,t}| = 2.0.
$$

This is important. Baseline I already imposed factor neutrality, but Baseline II makes that neutrality operational inside a more active, broader portfolio. The factor hedge is not just descriptive; it is explicitly built into the target weights before PnL is computed.

---

### 9. Transaction costs and net PnL

Let $w_t$ denote the full target weight vector, including factor hedge legs, and let $w_{t-1}$ denote the previous portfolio. Turnover is measured by

$$
\text{Turnover}_t = \sum_j |w_{j,t} - w_{j,t-1}|.
$$

With transaction cost rate $c = 5$ basis points, trading cost is

$$
\text{Cost}_t = c \cdot \text{Turnover}_t.
$$

If $\text{PnL}_t^{\text{gross}}$ denotes next-period gross portfolio PnL, then the reported strategy return is

$$
\text{PnL}_t^{\text{net}} = \text{PnL}_t^{\text{gross}} - \text{Cost}_t.
$$

This keeps the backtest economically meaningful and prevents the Sharpe ratio from being driven by unrealistic rebalancing.

---

### 10. What changed from Baseline I

Relative to Baseline I, the structural changes are clear.

Baseline I used a static OU-implied equilibrium, cross-sectional equilibrium centering, and a signal of the form

$$
s_i^{(1)} = -\frac{m_i^{adj}}{\sigma_{eq,i}},
$$

with $K = 15$, an explicit $\kappa > 14$ filter, entry threshold $1.75$, and average portfolio breadth around 15 names. Baseline II instead uses $K = 9$, a 160-day regression window, a Kalman-updated fair value $m_{i,t}$, a deviation-based signal

$$
s_{i,t}^{(2)} = \frac{X_{i,t} - m_{i,t}}{\sigma_{dyn,i,t}},
$$

and a much broader factor-hedged book. In short, Baseline I estimated where equilibrium should be from a rolling window; Baseline II estimates where equilibrium is now by sequential updating.

---

### 11. Empirical results

The final Baseline II implementation reports annual return $0.0918$, annual volatility $0.0999$, Sharpe ratio $0.9191$, maximum drawdown $-0.0984$, and Calmar ratio $0.9332$. Trade-quality statistics show a win rate $0.5137$, average win $0.00425$, average loss $-0.00376$, and profit factor $1.2001$. Portfolio statistics show average longs $32.20$, average shorts $35.95$, average total positions $68.15$, average turnover $0.1130$, and active trading on essentially all days.

By comparison, your Baseline I write-up reports Sharpe $\approx 0.60$, annual return $\approx 5.3\%$, annual volatility $\approx 8.9\%$, maximum drawdown $\approx -18.6\%$, profit factor $\approx 1.11$, average positions $\approx 15$, and turnover $\approx 0.35$. The main empirical takeaway is therefore not only that Baseline II improves risk-adjusted performance, but also that it does so while running a much broader and more stable book with materially lower average turnover. That is exactly what one would hope to see if the Kalman component is improving fair-value estimation rather than merely amplifying noise.

---

### 12. Evaluation, robustness testing, and parameter selection

The draft evaluation code adds a stronger research-style validation layer around the final model. It defines a full test suite rather than relying on headline Sharpe alone. Specifically, the diagnostics include train/test split Sharpe, noise injection, random subsampling, one-period lagged execution, higher transaction-cost stress, correlation of returns with turnover, signal distribution checks, parameter stability under perturbation, rolling risk-adjusted performance, and monotonicity of returns across signal bins.

For the reported parameter set, the draft outputs a train/test Sharpe pair of approximately $(0.848, 0.802)$, suggesting only modest degradation out of sample. The noise-stressed Sharpe is $0.891$, the lagged-execution Sharpe is $0.840$, the triple-cost Sharpe falls to $0.558$, the turnover-return correlation is near zero at $0.0089$, the signal distribution has mean $0.00694$ and standard deviation $0.08246$, the stability test under a small $\delta_A$ perturbation gives $0.832$, and the rolling performance proxy is $0.997$. These tests collectively suggest that the strategy is not entirely fragile, but that it remains meaningfully sensitive to implementation frictions, especially transaction costs and subsample choice.

The monotonicity test is also informative. The average returns across signal quintiles are not perfectly ordered, which means the signal is useful but not fully monotone in a cross-sectional sense. That is a subtle but important research result: the model appears to work more as a thresholded mean-reversion engine than as a perfectly ranked predictor. In other words, extreme deviations are tradable, but intermediate deviations do not line up in a clean linear fashion.

Your draft also includes an Optuna-based Bayesian optimization framework for broader hyperparameter search. That framework searches over $N_{PCA} \in [7,12]$, $OU\_WINDOW \in [120,200]$, entry threshold in $[1.25,2.00]$, long and short exit bands, $KAPPA\_MIN \in [6,16]$, $\delta_A \in [0.996,0.999]$, and $OBS\_VAR\_SCALE \in [0.8,2.5]$, with an objective that can be either pure Sharpe or a penalized score of the form

$$
\text{Score} = \text{Sharpe} - 0.50|\text{DD}| - 0.05 \cdot \text{Turnover},
$$

plus viability filters such as minimum average positions. That is exactly the kind of parameter-selection language that makes the paper sound more serious, because it shows that model choice was not ad hoc but constrained by performance quality, drawdown control, and implementability.

---

### 13. Interpretation

The economic interpretation of Baseline II is straightforward. PCA strips out common sector-wide and market-wide co-movement. The residualized cumulative spread then measures stock-specific dislocation. The Kalman filter estimates where that stock-specific spread should currently be centered, conditional on new information. The OU block measures whether the spread has historically behaved like a mean-reverting object. Finally, the signal compares current dislocation to its recent scale and only trades when that deviation is large enough to plausibly represent temporary mispricing rather than noise.

From a research perspective, the main conclusion is that Baseline II improves upon Baseline I not by changing the core statistical-arbitrage idea, but by improving the state estimation problem inside that idea. The model still says “buy residual underpricing, short residual overpricing.” What has changed is the estimate of underpricing and overpricing itself. Static equilibrium has been replaced by filtered equilibrium, and static normalization has been replaced by partially dynamic normalization. That is the correct conceptual story for the paper.
